# Benchmark against vLLM on an NVIDIA GPU

Runs the full comparison ladder on one workload and one GPU: Hugging Face baseline, our engine (static batching, continuous batching, + paged KV cache, + prefix caching), and **vLLM** twice: once with the KV memory vLLM picks for itself, and once capped at the same KV budget our engine gets.

**Which GPU:** an **L4, A10 or A100** (Colab: Runtime → Change runtime type → L4 / A100). The published numbers come from an L4. A **T4** (Kaggle and free Colab) works for our engine in `float16`; recent vLLM releases may not support it.

**Kaggle:** Settings → Accelerator → GPU, and turn Internet on.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
!git clone https://github.com/devt-code/llm-serving-engine.git
%cd llm-serving-engine
!pip install -q -r requirements.txt
!pip install -q vllm
# vLLM installs its own PyTorch build, and both engines run on it. Colab's preinstalled torchaudio is built
# for a different CUDA version and breaks Hugging Face's model import; nothing here needs it.
!pip uninstall -y -q torchaudio

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen3-0.6B', local_dir='models/qwen3-0.6b')

In [ ]:
import torch
major, _ = torch.cuda.get_device_capability()
DTYPE = 'bfloat16' if major >= 8 else 'float16'
print('using', DTYPE)

## 1. Our engine and the Hugging Face baseline
Same model, prompts and output lengths for every configuration. Our engine's KV memory is fixed at the budget printed on the first line.

In [ ]:
!python bench/ladder.py --dtype $DTYPE --only hf,static,continuous,paged,prefix --out bench/results/ladder_gpu.json

## 2. vLLM
Two runs: vLLM with its default memory setting (`gpu_memory_utilization=0.5`, so it sizes its own KV cache), and vLLM with its KV cache capped at our budget. Each runs in its own process, so the first frees the GPU before the second starts.

In [ ]:
!python bench/ladder.py --dtype $DTYPE --only vllm --out bench/results/ladder_gpu.json
!python bench/ladder.py --dtype $DTYPE --only vllm_kv --out bench/results/ladder_gpu.json

In [ ]:
import json
r = json.load(open('bench/results/ladder_gpu.json'))
print(r['gpu'], r['dtype'], 'torch', r['torch'])
by = {x['config']: x for x in r['results']}
for x in r['results']:
    print(f"{x['label']:<36} {x['output_tok_per_s']:8.1f} tok/s")
ours = max(by[c]['output_tok_per_s'] for c in ('static', 'continuous', 'paged', 'prefix') if c in by)
for c in ('vllm_kv', 'vllm'):
    if c in by:
        print(f"Our best = {100 * ours / by[c]['output_tok_per_s']:.0f}% of {by[c]['label']}")

In [ ]:
!python -m pytest tests/test_cuda.py -v

Download `bench/results/ladder_gpu.json` (Kaggle: Output tab; Colab: Files panel) and commit it, or paste the printed table back into the chat.